# DQ SRC MOBILE

Визуализация последнего прогона `dq-src-mobile` (логи `DQ_SRC_MOBILE`): покрытие витрин, cross-mart микс и gate `stg_contract`.

Перед запуском: `uv run mobile build-src-mobile`, затем `uv run mobile dq-src-mobile`.

In [ ]:
from __future__ import annotations

import pandas as pd
from IPython.display import display

from mobile.project_paths import PROJECT_ROOT

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 120)

ROOT = PROJECT_ROOT
TAG = "DQ_SRC_MOBILE"
BOUNDARY = "cdr.coverage"

In [ ]:
from mobile.pipelines.nb.common import checks_by_status, load_dq_dashboard

try:
    dq_logs, latest, meta = load_dq_dashboard(ROOT, tag=TAG, boundary_check=BOUNDARY)
except (FileNotFoundError, ValueError) as exc:
    print(f"Нет DQ-логов для {TAG}: {exc}")
    latest = meta = None
else:
    print("Последний прогон:", meta)
    display(checks_by_status(latest))

In [ ]:
from mobile.pipelines.nb.common import failed_warning_table, metrics_wide_table

if latest is None:
    pass
else:
    display(pd.DataFrame([meta]))
    print("\n--- failed / warning ---")
    display(failed_warning_table(latest))
    print("\n--- все метрики (плоская таблица) ---")
    wide = metrics_wide_table(latest)
    display(wide.head(80))
    if len(wide) > 80:
        print(f"... всего строк metrics: {len(wide)}")

## Визуализации DQ (только метрики из лога)

Все графики — поля `metrics` из JSON-логов `DQ_SRC_MOBILE` (без чтения parquet). Прогон в ноутбуке режется по check `cdr.coverage`.

In [ ]:
import matplotlib.pyplot as plt

from mobile.pipelines.nb.common import (
    mobile_coverage_frame,
    mobile_stg_gate_frame,
    mobile_traffic_mix_frame,
    render_src_mobile_dq_marts,
    render_src_mobile_dq_overview,
)

if latest is None:
    print("Пропуск графиков: нет логов DQ.")
else:
    fig = render_src_mobile_dq_overview(latest)
    plt.show()
    fig = render_src_mobile_dq_marts(latest)
    plt.show()
    cov = mobile_coverage_frame(latest)
    mix = mobile_traffic_mix_frame(latest, day=True)
    gates = mobile_stg_gate_frame(latest)
    print(f"Витрин в coverage: {len(cov)}")
    if not cov.empty:
        print(cov[["label", "row_count_total", "parquet_files_scanned"]].to_string(index=False))
    if not mix.empty:
        print(f"\nDay traffic mix rows: {int(mix['rows'].sum()):,}")
    print(f"STG gate checks: {len(gates)}")